# LangChain XML Output Parser Reference

Developer-facing statements defined in `langchain_core.output_parsers.xml`.

# `XML_FORMAT_INSTRUCTIONS`

Template used by `XMLOutputParser.get_format_instructions()` to request XML output.

```python
XML_FORMAT_INSTRUCTIONS = """The output should be formatted as a XML file.

1. Output should conform to the tags below.

2. If tags are not given, make them on your own.

3. Remember to always open and close all the tags.

As an example, for the tags ["foo", "bar", "baz"]:
1. String "<foo>\n <bar>\n <baz></baz>\n </bar>\n</foo>" is a well-formatted instance of the schema.

2. String "<foo>\n <bar>\n </foo>" is a badly-formatted instance.

3. String "<foo>\n <tag>\n </tag>\n</foo>" is a badly-formatted instance.

Here are the output tags:

```

{tags}

```"""
```

The `{tags}` placeholder is replaced with the parser's configured `tags` value.

---

# `XMLOutputParser: BaseTransformOutputParser[dict[str, Any]]`

Parses XML model output into nested dictionaries and supports incremental XML streaming.

## Fields

```python
tags: list[str] | None = None # Tags to request in generated XML output
encoding_matcher: re.Pattern[str] = re.compile(
    r"<([^>]*encoding[^>]*)>\n(.*)",
    re.MULTILINE | re.DOTALL,
) # Pattern used to remove an XML encoding declaration line
parser: Literal["defusedxml", "xml"] = "defusedxml" # XML parser implementation to use
```

`"defusedxml"` is the secure default. It requires the optional `defusedxml` package.

`"xml"` uses Python's standard-library XML parser and should be selected only when its XML-security implications are acceptable.

## Constructor

```python
XMLOutputParser(
    *,
    tags: list[str] | None = None, # Tags to request in generated XML output
    encoding_matcher: re.Pattern[str] = re.compile(
        r"<([^>]*encoding[^>]*)>\n(.*)",
        re.MULTILINE | re.DOTALL,
    ), # Pattern used to remove an XML encoding declaration line
    parser: Literal["defusedxml", "xml"] = "defusedxml", # XML parser implementation
) -> None
```

## Methods

### `get_format_instructions`

Returns XML output instructions containing the configured tag list.

```python
get_format_instructions(
    self,
) -> str # XML output-format instructions
```

The method formats `XML_FORMAT_INSTRUCTIONS` with `tags=self.tags`.

### `parse`

Parses complete XML text into a nested dictionary.

```python
parse(
    self,
    text: str, # Language-model output to parse
) -> dict[str, str | list[Any]] # Parsed XML representation
```

When `parser="defusedxml"`, the method uses `defusedxml.ElementTree`. If that package is unavailable, it raises `ImportError`.

When `parser="xml"`, it uses `xml.etree.ElementTree`.

The method searches for XML enclosed in triple backticks using the pattern ``r"```(xml)?(.*)```"`` with `re.DOTALL`. When found, only the content inside the fence is parsed.

If `encoding_matcher` matches, the method discards the matched encoding-containing first line and parses the remaining text. Surrounding whitespace is removed before parsing.

Malformed XML raises `OutputParserException`. The cleaned XML text is stored as `llm_output`.

For dictionary conversion:

- When the root contains non-whitespace text, the result is `{root_tag: root_text}`.
- Otherwise, each child is represented as a dictionary inside a list under the root tag.
- Leaf children use `{child_tag: child_text}`.
- Non-leaf children are converted recursively.

## Streaming behaviour

The inherited `transform()` and `atransform()` interfaces use this class's incremental XML parser.

Incoming strings are buffered. For message inputs, only string content is processed; messages with non-string content are ignored.

Text before the first XML opening tag is discarded. The stream parser uses XML start and end events and emits completed leaf elements as nested `AddableDict` values representing their current XML path.

Parent elements containing emitted children are not emitted separately. When the root closes, the parser can begin detecting another XML document later in the stream.

A parsing error caused by trailing content after a completed root is ignored when no XML path is active. Errors occurring while an element path remains open are re-raised.

Closing the streaming parser suppresses an XML parse error caused by incomplete XML remaining at the end of the stream.

---

# `nested_element`

Builds an addable nested dictionary for an XML element and its ancestor path.

```python
nested_element(
    path: list[str], # Ancestor tag path from the root toward the element
    elem: ET.Element, # XML element to represent
) -> Any # Nested AddableDict representation
```

When `path` is empty, the function returns:

```python
AddableDict({elem.tag: elem.text})
```

Otherwise, it recursively wraps the element under the first path component using a single-item list:

```python
AddableDict({path[0]: [nested_element(path[1:], elem)]})
```

In [ ]:
from collections.abc import AsyncIterator # Import the asynchronous iterator type
from xml.etree.ElementTree import Element # Import Element for the helper-function example

from langchain_core.exceptions import OutputParserException # Import the real LangChain parser exception
from langchain_core.messages import AIMessage, AIMessageChunk # Import real LangChain message classes
from langchain_core.output_parsers import XMLOutputParser # Import the real XML output parser
from langchain_core.output_parsers.xml import nested_element # Import the nested-element helper


parser = XMLOutputParser( # Create the XML parser
    tags=["person", "name", "age", "city"], # Define the expected XML tags
    parser="xml", # Use Python's built-in XML parser
) # Finish creating the parser

print("Format instructions:") # Display a heading
print(parser.get_format_instructions()) # Display XML output instructions

xml_text = """<person>
    <name>Saad</name>
    <age>22</age>
    <city>Delhi</city>
</person>""" # Create complete XML output

parsed_data = parser.parse(xml_text) # Parse the complete XML text
print("\nParsed XML:", parsed_data) # Display the nested dictionary

invoked_data = parser.invoke( # Parse through the runnable interface
    xml_text, # Provide XML text
    config={"run_name": "parse_person_xml"}, # Name the parser run
) # Finish invoking the parser

print("Invoke result:", invoked_data) # Display the runnable result

message = AIMessage( # Create a real AI message containing XML
    content="<product><name>Laptop</name><price>59999</price></product>", # Provide XML content
) # Finish creating the message

message_result = parser.invoke(message) # Parse the AI message
print("Message result:", message_result) # Display the parsed message content

fenced_xml = """```xml
<course>
    <name>Python</name>
    <level>Beginner</level>
</course>
```""" # Create XML inside a Markdown code block

fenced_result = parser.parse(fenced_xml) # Parse XML from the code block
print("Fenced XML result:", fenced_result) # Display the parsed fenced XML

async_result = await parser.ainvoke( # Parse asynchronously in Jupyter
    "<book><title>Clean Code</title><author>Robert Martin</author></book>", # Provide XML text
    config={"run_name": "parse_book_xml"}, # Name the asynchronous parser run
) # Finish asynchronous parsing

print("Async result:", async_result) # Display the asynchronous result

stream_chunks = iter([ # Create synchronous XML chunks
    "<person><name>Sa", # Provide the opening XML and partial name
    "ad</name><age>22</age>", # Complete the name and age elements
    "<city>Delhi</city></person>", # Complete the city and root elements
]) # Finish creating the chunk iterator

print("\nSynchronous streaming:") # Display a heading

for parsed_chunk in parser.transform(stream_chunks): # Parse XML chunks incrementally
    print(parsed_chunk) # Display each completed leaf element


async def generate_xml_chunks() -> AsyncIterator[AIMessageChunk]: # Define an asynchronous XML stream
    yield AIMessageChunk(content="<person><name>Aman</name>") # Yield the first XML chunk
    yield AIMessageChunk(content="<age>25</age>") # Yield the second XML chunk
    yield AIMessageChunk(content="<city>Mumbai</city></person>") # Yield the final XML chunk


print("\nAsynchronous streaming:") # Display a heading

async for parsed_chunk in parser.atransform(generate_xml_chunks()): # Parse message chunks asynchronously
    print(parsed_chunk) # Display each completed leaf element

leaf = Element("skill") # Create a real XML element
leaf.text = "Python" # Assign text to the XML element

nested_value = nested_element( # Build a nested dictionary for the XML path
    ["person", "skills"], # Provide the ancestor tag path
    leaf, # Provide the leaf XML element
) # Finish building the nested value

print("\nNested element:", nested_value) # Display the helper-function result

try: # Start malformed XML error handling
    parser.parse("<person><name>Saad</person>") # Parse XML with a missing closing tag
except OutputParserException as error: # Catch the LangChain parser exception
    print("\nXML parsing error:", error) # Display the parsing error
    print("Invalid XML output:", error.llm_output) # Display the malformed XML text